# Unknown $\sigma$: the Student t-Distribution

Every confidence interval so far leaned on a magically known population standard deviation. Drop that crutch — as reality usually forces you to — and the fix looks tiny: use the *sample* standard deviation $s$ instead. But the standardized quantity $(\bar{X} - \mu)/(s/\sqrt{n})$ is then no longer normal: it follows the **Student t-distribution**, a bell curve with **fatter tails**. Keep using the z-score with it and your intervals are quietly too narrow; the honest fix is a **t-score** from the t-distribution with $n - 1$ **degrees of freedom** — which melts back into the normal as $n$ grows, exactly because $s$ melts into $\sigma$. It is based on the eighth video of Lesson 7 (Confidence Intervals).

## 1. Introduction

"Assume $\sigma$ is known" was always a placeholder. This notebook removes it: what changes (the scale $s$ and the score $t$), what doesn't (the whole recipe structure), and why the correction shrinks as samples grow.

After completing this notebook you will be able to:

- Explain why substituting $s$ for $\sigma$ changes the sampling distribution from normal to Student t.
- Describe the t-distribution's shape — fatter tails — and what extra uncertainty they encode.
- Write both interval formulas side by side: $\bar{x} \pm z_{1-\alpha/2}\,\sigma/\sqrt{n}$ vs $\bar{x} \pm t_{1-\alpha/2}\,s/\sqrt{n}$.
- Use degrees of freedom ($n - 1$) and show the t-distribution converging to the normal as they grow.
- Demonstrate by simulation that z-scores with $s$ under-cover, and t-scores repair it.

## 2. Theory

### The last assumption falls

The interval formula $\bar{x} \pm z_{1-\alpha/2}\,\sigma/\sqrt{n}$ works because, with $\sigma$ known, the standardized sampling quantity

$$
Z = \frac{\bar{X} - \mu}{\sigma/\sqrt{n}}
$$

is exactly standard normal — that is what justifies a critical value read off the normal curve. But more often than not you *don't* know $\sigma$. No problem: you took a sample, so you can compute the **sample standard deviation** $s$ and use it instead:

$$
T = \frac{\bar{X} - \mu}{s/\sqrt{n}}
$$

### A new distribution

The swap is not free. $T$ is **no longer normally distributed** — it follows the **Student t-distribution**: very similar to the normal, but with **much fatter tails**. A point sampled from it is more likely to land far from the center than one sampled from the normal.

Where do the fat tails come from? $s$ is itself random. Some samples happen to have a small $s$, which *inflates* $T$ — occasionally enormously. The t-distribution prices in this second source of uncertainty.

### The scaling fix: t-scores

If you replaced $\sigma$ with $s$ in the margin of error but kept the critical value $z_{1-\alpha/2}$, you would be wrong — that $z$ is defined over the *normal* distribution, and your quantity no longer follows it. The fix: take the critical value from the distribution you actually have. The **t-score** $t_{1-\alpha/2}$ plays exactly the role $z$ played, but on the t-distribution — and since the t-curve's tails are fatter, $t_{1-\alpha/2} > z_{1-\alpha/2}$: honest intervals get a little wider.

### Side by side

| | $\sigma$ known | $\sigma$ unknown (the usual case) |
|---|---|---|
| scale | population $\sigma$ | sample $s$ |
| distribution of the statistic | normal | Student t |
| critical value | $z_{1-\alpha/2}$ | $t_{1-\alpha/2}$ |
| interval | $\bar{x} \pm z_{1-\alpha/2}\,\dfrac{\sigma}{\sqrt{n}}$ | $\bar{x} \pm t_{1-\alpha/2}\,\dfrac{s}{\sqrt{n}}$ |

### Degrees of freedom

There isn't one t-distribution but a family, indexed by the **degrees of freedom**:

$$
\text{df} = n - 1
$$

The degrees of freedom set how tall and tight the curve is. At df $= 1$ the tails are huge; by df $= 5$ they've slimmed; at df $= 10$ the curve hugs the normal closely — and as df grows it converges to the normal outright. That makes sense: bigger samples make $s$ a better and better estimate of $\sigma$, and if you *knew* $\sigma$ you'd be back on the normal. (The odd name, "Student," was the pen name William Sealy Gosset published under while working at the Guinness brewery.)

## 3. Worked Examples

### Example 1: the wrong interval and the right one

A sample of $n = 10$ islanders: $\bar{x} = 170$ cm, $s = 25$ cm, 95% confidence, df $= 9$.

- **Wrong** (z with $s$): $170 \pm 1.96 \times 25/\sqrt{10} = 170 \pm 15.5 = [154.5, 185.5]$
- **Right** (t with $s$): $t_{0.975,\,9} = 2.262$, so $170 \pm 2.262 \times 25/\sqrt{10} = 170 \pm 17.9 = [152.1, 187.9]$

The z version is ~14% too narrow. It *looks* more precise — but the precision is fake, paid for by covering $\mu$ less often than promised.

### Example 2: how the t-score falls with df

The 97.5% critical values ($z$ equivalent: 1.960):

| df | 1 | 4 | 9 | 29 | 99 | $\infty$ |
|---|---|---|---|---|---|---|
| $t_{0.975}$ | 12.706 | 2.776 | 2.262 | 2.045 | 1.984 | 1.960 |

At df $= 1$ the fat tails demand a margin six times the normal one; by df $\approx 30$ the premium is under 5%, and it never quite reaches zero.

### Example 3: which score, which scale?

- $\sigma$ known, $n = 15$: normal, $z$, scale $\sigma$ — sample size is irrelevant to the *choice*.
- $\sigma$ unknown, $n = 15$: t with df $= 14$, scale $s$.
- $\sigma$ unknown, $n = 200$: formally t with df $= 199$, but $t_{0.975,199} = 1.972 \approx z$ — at this size the distinction is cosmetic.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project). We build the t-distribution from its density — no statistics library needed: the critical value is found by numerically integrating the pdf and bisecting on the tail area.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from math import gamma, sqrt, pi

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(17)

def z_score(p):
    """Normal critical value z_p (probability p to the left)."""
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

def t_pdf(t, df):
    """Density of the Student t-distribution with df degrees of freedom."""
    coeff = gamma((df + 1) / 2) / (sqrt(df * pi) * gamma(df / 2))
    return coeff * (1 + np.asarray(t) ** 2 / df) ** (-(df + 1) / 2)

def t_score(p, df):
    """t critical value t_p by bisection on the numerically integrated pdf."""
    target = p - 0.5                       # area between 0 and the answer
    lo, hi = 0.0, 700.0
    for _ in range(60):
        mid = (lo + hi) / 2
        area = np.trapezoid(t_pdf(np.linspace(0, mid, 4000), df),
                            np.linspace(0, mid, 4000))
        lo, hi = (mid, hi) if area < target else (lo, mid)
    return (lo + hi) / 2

### The critical-value table, computed — and checked by simulation

The video's df story in numbers, then a brute-force check: generate a million $T = (\bar{X} - \mu)/(s/\sqrt{n})$ statistics from real samples and see where their 97.5% quantile actually sits.

In [ ]:
print(f"{'df':>5s} {'t_0.975':>9s}   (z_0.975 = {z_score(0.975):.3f})")
for df in (1, 4, 9, 29, 99, 199):
    print(f"{df:>5d} {t_score(0.975, df):>9.3f}")

n = 10
samples = rng.normal(170, 25, size=(1_000_000, n))
t_stats = (samples.mean(axis=1) - 170) / (samples.std(axis=1, ddof=1) / np.sqrt(n))
print(f"\nempirical 97.5% quantile of T at n = 10: {np.quantile(t_stats, 0.975):.3f}")
print(f"computed t_0.975 with df = 9:            {t_score(0.975, 9):.3f}")
print(f"the z-score would have said:             {z_score(0.975):.3f}  (too small)")

### Example 1's two intervals, and the honest one's price

In [ ]:
x_bar, s, n = 170.0, 25.0, 10
se = s / np.sqrt(n)

m_wrong = z_score(0.975) * se
m_right = t_score(0.975, n - 1) * se
print(f"wrong (z with s): 170 +/- {m_wrong:.1f} = [{x_bar - m_wrong:.1f}, {x_bar + m_wrong:.1f}]")
print(f"right (t with s): 170 +/- {m_right:.1f} = [{x_bar - m_right:.1f}, {x_bar + m_right:.1f}]")
print(f"the honest interval is {m_right / m_wrong - 1:.0%} wider -- the price of not knowing sigma")

### Coverage: the z-shortcut quietly breaks the promise

10,000 surveys of $n = 10$, each interval built from its own $\bar{x}$ *and its own $s$*. With the z-score the "95%" intervals cover far less; the t-score restores the promise.

In [ ]:
MU_TRUE, SIGMA_TRUE = 170.0, 25.0
n = 10
surveys = rng.normal(MU_TRUE, SIGMA_TRUE, size=(10_000, n))
x_bars = surveys.mean(axis=1)
s_vals = surveys.std(axis=1, ddof=1)
se_vals = s_vals / np.sqrt(n)

for name, crit in [("z-score 1.960 (wrong)", z_score(0.975)),
                   ("t-score 2.262 (right)", t_score(0.975, n - 1))]:
    covered = (np.abs(x_bars - MU_TRUE) <= crit * se_vals).mean()
    print(f"{name}: coverage = {covered:.1%}")

## 5. Visualizations

### The video's picture: t-distributions closing in on the normal

In [ ]:
t_axis = np.linspace(-5, 5, 600)
normal_density = np.exp(-0.5 * t_axis**2) / np.sqrt(2 * np.pi)

plt.figure(figsize=(10, 5))
plt.plot(t_axis, normal_density, color="black", linewidth=2.5,
         label="standard normal")
for df, color in [(1, "tab:red"), (5, "tab:orange"), (10, "tab:green")]:
    plt.plot(t_axis, t_pdf(t_axis, df), color=color, linewidth=1.8,
             label=f"t with df = {df}")
plt.xlabel("t")
plt.ylabel("density")
plt.title("Student t: fatter tails at low degrees of freedom, normal in the limit")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The tails, up close

On a log scale the difference is dramatic: far from the center, the t with one degree of freedom carries orders of magnitude more probability than the normal.

In [ ]:
plt.figure(figsize=(10, 4.4))
plt.semilogy(t_axis, normal_density, color="black", linewidth=2.5,
             label="standard normal")
for df, color in [(1, "tab:red"), (5, "tab:orange"), (10, "tab:green")]:
    plt.semilogy(t_axis, t_pdf(t_axis, df), color=color, linewidth=1.8,
                 label=f"t with df = {df}")
plt.ylim(1e-6, 1)
plt.xlabel("t")
plt.ylabel("density (log scale)")
plt.title("Same curves, log scale: the fat tails are no rounding error")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The premium fades: $t_{0.975}$ vs degrees of freedom

In [ ]:
dfs = np.arange(1, 101)
t_crits = [t_score(0.975, int(df)) for df in dfs]

plt.figure(figsize=(9.5, 4.4))
plt.plot(dfs, t_crits, color="tab:blue", linewidth=2, label=r"$t_{0.975}$(df)")
plt.axhline(z_score(0.975), color="black", linestyle=":",
            label=r"$z_{0.975} = 1.96$ (the limit)")
for df in (1, 4, 9, 29):
    plt.plot(df, t_score(0.975, df), "ko", markersize=4)
plt.ylim(1.8, 4.5)
plt.xlabel("degrees of freedom (n - 1)")
plt.ylabel("97.5% critical value")
plt.title("As samples grow, s approaches sigma and the t-score approaches the z-score")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Where the fat tails come from: the randomness of $s$

Each simulated survey's statistic, colored by whether its $s$ underestimated $\sigma$. The extreme t-statistics almost all come from samples whose $s$ came out small — dividing by a too-small scale is what stretches the tails.

In [ ]:
n = 5
demo = rng.normal(0, 1, size=(40_000, n))
t_demo = demo.mean(axis=1) / (demo.std(axis=1, ddof=1) / np.sqrt(n))
s_demo = demo.std(axis=1, ddof=1)

plt.figure(figsize=(10, 4.6))
small_s = s_demo < 0.7
plt.scatter(t_demo[~small_s], s_demo[~small_s], s=2, alpha=0.25,
            color="tab:blue", label=r"$s \geq 0.7\sigma$")
plt.scatter(t_demo[small_s], s_demo[small_s], s=2, alpha=0.25,
            color="tab:red", label=r"$s < 0.7\sigma$ (underestimated)")
plt.xlim(-12, 12)
plt.xlabel("t-statistic")
plt.ylabel(r"sample standard deviation $s$ (true $\sigma$ = 1)")
plt.title("n = 5: the far tails belong to surveys whose s came out too small")
plt.grid(True, alpha=0.3)
plt.legend(markerscale=6, fontsize=9)
plt.show()

print(f"|T| > 3 happens in {np.mean(np.abs(t_demo) > 3):.2%} of samples "
      f"(normal would predict {2 * (1 - 0.99865):.2%});")
print(f"among those extremes, {np.mean(s_demo[np.abs(t_demo) > 3] < 1):.0%} "
      "had s below the true sigma.")

### The whole story on one page

Everything above in a single sheet: the t-distribution closing in on the normal, the critical values by degrees of freedom, the wrong and right intervals, and the coverage check.

<img src="img/07_t_distribution.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** For each scenario, name the distribution, the critical score, and the scale: (a) $\sigma$ known, $n = 15$; (b) $\sigma$ unknown, $n = 15$; (c) $\sigma$ unknown, $n = 200$.

**Exercise 2.** A sample of $n = 12$ has $\bar{x} = 55$ and $s = 6$. How many degrees of freedom, and what is the 95% confidence interval?

### Intermediate

**Exercise 3.** At $n = 4$ (df $= 3$), how much wider is the honest t-interval than the naive z-interval at 95%? And at 99%? What does this say about small samples with high confidence demands?

**Exercise 4.** Simulate the coverage of both recipes at $n = 5$ (as done above for $n = 10$). How badly does the z-shortcut miss now, and why is the damage worse than at $n = 10$?

### Advanced

**Exercise 5.** Find the smallest sample size at which the t-score is within 1% of the z-score at the 95% level — the point where the classic "just use z for large n" rule of thumb becomes honest to within a percent.

**Exercise 6.** *One real dataset, start to finish.* Simulate a single survey of $n = 20$ heights from Statistopia ($\mu = 170$, $\sigma = 25$ — both then forgotten), and produce the full 95% t-interval from the data alone: $\bar{x}$, $s$, df, $t$-score, margin, interval. Then peek at the truth and check. Compare with what the (illegitimate) known-$\sigma$ z-interval would have said.

## 7. Solutions

In [ ]:
# Exercise 1: which machinery
print("(a) sigma known, n = 15: normal distribution, z-score, scale sigma.")
print("(b) sigma unknown, n = 15: t-distribution (df = 14), t-score, scale s.")
print("(c) sigma unknown, n = 200: formally t (df = 199), scale s -- but")
print(f"    t = {t_score(0.975, 199):.3f} vs z = {z_score(0.975):.3f}: in practice z is fine.")

In [ ]:
# Exercise 2: n = 12
df12 = 12 - 1
t12 = t_score(0.975, df12)
m12 = t12 * 6 / np.sqrt(12)
print(f"df = {df12}, t_0.975 = {t12:.3f}")
print(f"interval: 55 +/- {m12:.2f} = [{55 - m12:.2f}, {55 + m12:.2f}]")

In [ ]:
# Exercise 3: the small-sample premium
for level in (0.95, 0.99):
    p = 1 - (1 - level) / 2
    ratio = t_score(p, 3) / z_score(p)
    print(f"{level:.0%}: t(df=3) = {t_score(p, 3):.3f} vs z = {z_score(p):.3f} "
          f"-> {ratio - 1:.0%} wider")
print("\nThe premium grows with the confidence demand: tiny samples plus high")
print("confidence is exactly where the fat tails bite hardest.")

In [ ]:
# Exercise 4: coverage at n = 5
n5 = 5
sv = rng.normal(MU_TRUE, SIGMA_TRUE, size=(20_000, n5))
xb5, se5 = sv.mean(axis=1), sv.std(axis=1, ddof=1) / np.sqrt(n5)
for name, crit in [("z 1.960", z_score(0.975)), ("t 2.776", t_score(0.975, n5 - 1))]:
    print(f"{name}: coverage = {(np.abs(xb5 - MU_TRUE) <= crit * se5).mean():.1%}")
print("\nWorse than at n = 10: fewer observations make s noisier, the tails fatter,")
print("and the z-shortcut's blind spot bigger.")

In [ ]:
# Exercise 5: when is z within 1%?
df = 1
while t_score(0.975, df) > 1.01 * z_score(0.975):
    df += 1
print(f"t_0.975 first comes within 1% of z at df = {df} (n = {df + 1}):")
print(f"t = {t_score(0.975, df):.4f} vs 1.01 * z = {1.01 * z_score(0.975):.4f}")

In [ ]:
# Exercise 6: one survey, the honest way
rng6 = np.random.default_rng(2)                # a fresh, reproducible draw
survey = rng6.normal(170, 25, size=20)         # nature draws; we now forget mu, sigma
x_bar6 = survey.mean()
s6 = survey.std(ddof=1)
df6 = len(survey) - 1
t6 = t_score(0.975, df6)
m6 = t6 * s6 / np.sqrt(len(survey))
print(f"x-bar = {x_bar6:.2f}, s = {s6:.2f}, df = {df6}, t_0.975 = {t6:.3f}")
print(f"t-interval: [{x_bar6 - m6:.1f}, {x_bar6 + m6:.1f}]  (margin {m6:.2f})")

m6z = z_score(0.975) * 25 / np.sqrt(20)        # the interval only an oracle could build
print(f"known-sigma z-interval would be: [{x_bar6 - m6z:.1f}, {x_bar6 + m6z:.1f}]")
print(f"truth check: mu = 170 inside the t-interval? "
      f"{x_bar6 - m6 <= 170 <= x_bar6 + m6}")
print("\nNote the two intervals differ twice over: s replaces sigma AND t replaces z.")

## 8. Conclusion

- With $\sigma$ unknown — the usual case — use the sample standard deviation $s$; the standardized statistic $(\bar{X} - \mu)/(s/\sqrt{n})$ then follows the **Student t-distribution**, not the normal.
- The t-curve's **fatter tails** price in the randomness of $s$ itself: samples with a lucky small $s$ inflate the statistic, so extremes are more common than the normal admits.
- The repair is a matched critical value: $\bar{x} \pm t_{1-\alpha/2}\, s/\sqrt{n}$ with **df $= n - 1$**. Keeping the z-score with $s$ produces intervals that are too narrow and quietly under-cover (91.9% instead of 95% at $n = 10$; worse at $n = 5$).
- As df grows, $s \to \sigma$ and the t-distribution converges to the normal: the 97.5% t-score falls from 12.7 (df 1) through 2.26 (df 9) toward 1.96, and past $n \approx 30$–200 the z rule of thumb is honest to within a percent.

This closes the gap the whole lesson left open — confidence intervals now work with nothing but the sample in your hands.